# Data quality deep diveHow the expectation engine evaluates rules, splits valid from quarantined rows andrecords the outcome.

In [ ]:
import syssys.path.insert(0, "../src")from lakehouse.config import load_envfrom lakehouse.quality.expectations import load_suitefrom lakehouse.quality.runner import DataQualityRunnerfrom lakehouse.spark_session import get_sparkspark = get_spark("quality", load_env("local"))

## The shipped suites

In [ ]:
for name in ("orders", "customers", "products"):    suite = load_suite(name)    print(f"\n{name} ({len(suite.expectations)} rules)")    for e in suite.expectations:        print(f"  [{e.action.value:>10}] {e.name:<28} {e.expr[:60]}")

## Evaluate a suite against Silver

In [ ]:
suite = load_suite("orders")df = spark.table("silver.silver_orders")valid, quarantined, report = DataQualityRunner(spark).evaluate(df, suite, "orders")print(f"total={report.total_rows:,}  valid={report.valid_rows:,}  "      f"quarantined={report.quarantined_rows:,}  pass_rate={report.pass_rate:.2%}")

## Per-rule outcomes

In [ ]:
import pandas as pdpd.DataFrame([r.as_dict() for r in report.results]).sort_values("failed_rows", ascending=False)

## NULL-safetyIn SQL `length(NULL) > 1` evaluates to `NULL`, not `false`. A naive`WHERE predicate` / `WHERE NOT predicate` split lets such rows escape *both* sides.Each rule is wrapped in `coalesce(<expr>, false)` and the split uses those flagcolumns, so a NULL predicate counts as a failure.

In [ ]:
from lakehouse.quality.expectations import Action, Expectation, ExpectationSuitedemo = spark.createDataFrame([(None,), ("ok",)], "v string")s = ExpectationSuite("demo", [Expectation("len_ok", "length(v) > 1", Action.QUARANTINE)])v, q, rep = DataQualityRunner(spark).evaluate(demo, s, "demo")print(f"valid={v.count()}  quarantined={q.count()}   <- the NULL row is quarantined")

## Quality trend over time

In [ ]:
spark.sql('''    SELECT dataset, rule, sum(failed_rows) AS failed, sum(total_rows) AS total,           round(1 - sum(failed_rows)/nullif(sum(total_rows),0), 4) AS pass_rate    FROM ops.dq_results    GROUP BY ALL    ORDER BY pass_rate ASC''').show(truncate=False)